In [4]:
import os
import requests
from dotenv import load_dotenv

# Load private settings from the .env file (never uploaded to GitHub)
load_dotenv()
HEADERS = {"User-Agent": os.getenv("SEC_USER_AGENT")}
print("User-Agent loaded:", HEADERS["User-Agent"] is not None)

CIK = "0001436126"  # Mistras Group (10 digits, with leading zeros)
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"

response = requests.get(url, headers=HEADERS, timeout=30)
print("Status code:", response.status_code)

data = response.json()
print("Company:", data["entityName"])

User-Agent loaded: True
Status code: 200
Company: Mistras Group, Inc.


In [5]:
revenue_facts = data["facts"]["us-gaap"]["RevenueFromContractWithCustomerExcludingAssessedTax"]["units"]["USD"]

print("Number of revenue entries:", len(revenue_facts))
print("First entry:", revenue_facts[0])

Number of revenue entries: 187
First entry: {'start': '2015-06-01', 'end': '2015-12-31', 'val': 427913000, 'accn': '0001436126-19-000014', 'fy': 2018, 'fp': 'FY', 'form': '10-K', 'filed': '2019-03-18'}


In [6]:
import pandas as pd

# Turn the list of entries into a table
rev = pd.DataFrame(revenue_facts)

# Convert date text into real dates so we can do date math
rev["start"] = pd.to_datetime(rev["start"])
rev["end"] = pd.to_datetime(rev["end"])
rev["filed"] = pd.to_datetime(rev["filed"])

# Length of each period in days
rev["days"] = (rev["end"] - rev["start"]).dt.days

# Rules 1 and 2: annual reports only, and only full-year periods
annual = rev[(rev["form"] == "10-K") & (rev["days"].between(350, 380))]

# Rule 3: for each period end date, keep the most recently filed version
annual = annual.sort_values("filed").drop_duplicates(subset="end", keep="last")

# Rule 4: convert to thousands, and keep the useful columns
annual["revenue_000s"] = annual["val"] / 1000
annual = annual[["end", "revenue_000s", "accn", "filed", "days"]].sort_values("end")

annual

,end,revenue_000s,accn,filed,days
1,2016-05-31,719181.0,0001436126-19-000014,2019-03-18,365
2,2016-12-31,684762.0,0001436126-19-000014,2019-03-18,365
12,2017-12-31,700970.0,0001436126-20-000018,2020-03-27,364
32,2018-12-31,742354.0,0001436126-21-000013,2021-03-16,364
56,2019-12-31,748586.0,0001436126-22-000013,2022-03-14,364
84,2020-12-31,592571.0,0001436126-23-000014,2023-03-15,365
115,2021-12-31,677131.0,0001436126-24-000025,2024-03-11,364
142,2022-12-31,687373.0,0001436126-25-000026,2025-03-11,364
161,2023-12-31,705473.0,0001628280-26-016765,2026-03-11,364
174,2024-12-31,729640.0,0001628280-26-016765,2026-03-11,365


In [7]:
# Revenue from the Excel answer key ($000s), typed from the 10-Ks by hand
answer_key = {2025: 724024, 2024: 729640, 2023: 705473, 2022: 687373}

for year, expected in answer_key.items():
    actual = annual.loc[annual["end"].dt.year == year, "revenue_000s"].iloc[0]
    result = "MATCH" if actual == expected else "MISMATCH"
    print(year, "| answer key:", expected, "| API:", actual, "|", result)

2025 | answer key: 724024 | API: 724024.0 | MATCH
2024 | answer key: 729640 | API: 729640.0 | MATCH
2023 | answer key: 705473 | API: 705473.0 | MATCH
2022 | answer key: 687373 | API: 687373.0 | MATCH
